In [5]:
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.mask import mask
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT
from pystac_client import Client
from shapely.geometry import mapping

In [2]:
def download_clip_sentinel(
    boundary_file,
    start_date,
    end_date,
    out_path,
    max_cc=10,
    bands=["red", "nir", "swir16"],
    resolution=20.0,
):
    boundary = gpd.read_file(boundary_file)
    boundary_wgs84 = boundary.to_crs("EPSG:4326")

    search_geometry = mapping(boundary_wgs84.geometry.union_all())

    client = Client.open("https://earth-search.aws.element84.com/v1")
    search = client.search(
        collections=["sentinel-2-l2a"],
        intersects=search_geometry,
        datetime=f"{start_date}/{end_date}",
        query={"eo:cloud_cover": {"lt": max_cc}},
    )

    elements = list(search.items())
    if not elements:
        print("Nie znaleziono odpowiednich zdjęć dla zadanego obszaru i czasu.")
        return None

    best_scene = sorted(
        elements, key=lambda x: x.properties.get("eo:cloud_cover", 100)
    )[0]
    print(
        f"Wybrana scena: {best_scene.id} "
        f"(zachmurzenie: {best_scene.properties.get('eo:cloud_cover')}%)"
    )

    bands_data = {}
    out_meta = None

    target_crs = None
    target_transform = None
    target_width = None
    target_height = None
    shapes = None

    for band in bands:
        if band not in best_scene.assets:
            print(f"Pasmo {band} nie jest dostępne w wybranej scenie.")
            continue

        file_url = best_scene.assets[band].href

        with rasterio.open(file_url) as src:
            if target_crs is None:
                target_crs = src.crs
                boundary_utm = boundary.to_crs(target_crs)
                shapes = [geom.__geo_interface__ for geom in boundary_utm.geometry]

                minx, miny, maxx, maxy = boundary_utm.total_bounds
                target_width = int(np.ceil((maxx - minx) / resolution))
                target_height = int(np.ceil((maxy - miny) / resolution))
                target_transform = from_origin(minx, maxy, resolution, resolution)

            with WarpedVRT(
                src,
                crs=target_crs,
                transform=target_transform,
                width=target_width,
                height=target_height,
                resampling=Resampling.bilinear,
            ) as vrt:
                img, _ = mask(vrt, shapes, crop=False, nodata=0)
                bands_data[band] = img[0]
                if out_meta is None:
                    out_meta = src.meta.copy()
                    out_meta.update(
                        {
                            "driver": "GTiff",
                            "crs": target_crs,
                            "height": target_height,
                            "width": target_width,
                            "transform": target_transform,
                            "nodata": 0,
                        }
                    )

    if not bands_data:
        print("Nie udało się pobrać żadnego pasma.")
        return None
    
    for b in list(bands_data.keys()):
        arr = bands_data[b].astype(np.float32)
        arr[arr == 0] = np.nan
        bands_data[b] = arr
    if "nir" in bands_data and "red" in bands_data:
        nir, red = bands_data["nir"], bands_data["red"]
        denom = nir + red
        ndvi = np.where((denom != 0) & ~np.isnan(denom), (nir - red) / denom, np.nan)
        bands_data["NDVI"] = ndvi.astype(np.float32)
    else:
        print("Pominięto NDVI: brak pasma 'nir' lub 'red'.")
    if "nir" in bands_data and "swir16" in bands_data:
        nir, swir = bands_data["nir"], bands_data["swir16"]
        denom = nir + swir
        ndmi = np.where((denom != 0) & ~np.isnan(denom), (nir - swir) / denom, np.nan)
        bands_data["NDMI"] = ndmi.astype(np.float32)
    else:
        print("Pominięto NDMI: brak pasma 'nir' lub 'swir16'.")
    out_meta.update({"dtype": "float32", "nodata": np.nan})

    band_names = list(bands_data.keys())

    out_img = np.stack([bands_data[b] for b in band_names], axis=0)
    out_meta.update({"count": len(band_names)})

    with rasterio.open(out_path, "w", **out_meta) as dest:
        dest.write(out_img)
        for i, band in enumerate(band_names, start=1):
            dest.set_band_description(i, band)

    return out_path

In [13]:
boundary = "test/fields.geojson"
download_clip_sentinel(boundary_file=boundary, start_date="2022-06-01", end_date="2022-06-30", out_path="test/wycinek.tif")

Wybrana scena: S2A_33UWU_20220624_0_L2A (zachmurzenie: 0.000348%)


'test/wycinek.tif'